# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

We'll examine all record sets and their fields/columns. All entities are referenced by their `@id` (as per the Croissant specification).

In [ ]:
# List all record sets and display their fields and columns (by @id).
print("Available record sets and their fields (referenced by @id):\n")

recordset_list = []
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        print(f"Record set @id: {rs.id}")
        field_ids = [f.id for f in rs.fields] if hasattr(rs, 'fields') else []
        print(f"  Fields: {field_ids}")
        if hasattr(rs, 'columns'):
            column_ids = [c.id for c in rs.columns]
            print(f"  Columns: {column_ids}")
        print()
        recordset_list.append(rs.id)
else:
    print('No record sets found in the dataset schema.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We use the record set and field/column `@id` from the overview above.

Here, we pick the primary protein information record set (by its `@id`).

In [ ]:
# Extract data from every record set found in the schema
dataframes = {}
for record_set_id in recordset_list:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded records from record set {record_set_id} with shape: {dataframes[record_set_id].shape}")
    else:
        print(f"No records found for record set {record_set_id}")

# Select a primary record set for further analysis (the one with most rows or preferred by user)
if dataframes:
    main_record_set_id = max(dataframes, key=lambda k: dataframes[k].shape[0]) # the RS with most rows
    print(f"\nWill continue exploration on main record set: {main_record_set_id}")
    print(f"Columns: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print('No dataframes loaded.')

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping using only column `@id` as references, as per best Croissant practice.

Let's select a numeric column (e.g., protein abundance, peptide counts, or molecular weight) for filtering and normalization. We'll list all numeric columns first.

In [ ]:
# Identify numeric columns by dtype
df = dataframes[main_record_set_id]
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric columns in {main_record_set_id}:", numeric_cols)

# For demonstration, select the first numeric field (by @id)
if numeric_cols:
    numeric_field_id = numeric_cols[0]
else:
    print('No numeric fields found!')

# Set a threshold for filtering
threshold = df[numeric_field_id].mean() if numeric_cols else 0
filtered_df = df[df[numeric_field_id] > threshold] if numeric_cols else df.copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean value):")
display(filtered_df.head())

# Normalize the numeric column
if numeric_cols:
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try a group by if a suitable categorical column exists
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
group_field_id = None
# Prefer a column named like 'category', 'label', 'sample', or any non-numeric col
for c in cat_cols:
    if any(word in c.lower() for word in ['category','group','sample','label','type','condition']):
        group_field_id = c
        break
if not group_field_id and cat_cols:
    group_field_id = cat_cols[0]

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id} (first 5 rows):")
    display(grouped_df.head())
else:
    print('No suitable categorical field for grouping found.')

## 5. Visualization
Let's visualize the distribution of the chosen numeric column and explore relationship with the first categorical field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution
if numeric_cols:
    plt.figure(figsize=(8,6))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# Boxplot by group if available
if group_field_id and numeric_cols:
    plt.figure(figsize=(10,6))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and perform basic analysis on a mass spectrometry dataset of human proteins isolated from extracellular vesicles of mast cells, using the `mlcroissant` library in Python.

- The Croissant metadata schema allowed us to programmatically discover and reference record sets, fields, and columns by their `@id`.
- We loaded all available record sets, reviewed fields and columns, and extracted the largest table for analysis.
- Exploratory steps such as filtering, normalization, grouping, and visualization were performed using fields referenced by their `@id`.

You can build upon this analysis for downstream data integration, machine learning, or biochemical research!